In [1]:
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder.appName("spark-local-app")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/12 21:27:26 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/12 21:27:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/12 21:27:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
from pyspark.sql import Row
from datetime import datetime, timezone


customers: list[dict[str, Any]] = [
    {
        "id": 1,
        "first_name": "Иван",
        "last_name": "Иванов",
        "email": "ivanov@example.com",
        "phone_numbers": Row(mobile="+7 (999) 111-22-33", home="+7 (495) 111-22-33"),
        "is_customer": True,
        "amount_paid": 15450.50,
        "customer_from": datetime(2023, 5, 12, 10, 30, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2026, 1, 15, 14, 22, tzinfo=timezone.utc),
    },
    {
        "id": 2,
        "first_name": "Анна",
        "last_name": "Петрова",
        "email": "petrova@example.com",
        "phone_numbers": Row(mobile="+7 (999) 141-22-33", home="+7 (495) 111-12-33"),
        "is_customer": False,
        "amount_paid": 0.00,
        "customer_from": None,
        "last_updated_ts": datetime(2026, 2, 20, 9, 15, tzinfo=timezone.utc),
    },
    {
        "id": 3,
        "first_name": "Dmitry",
        "last_name": "Smith",
        "email": "d.smith@global-tech.com",
        "phone_numbers": Row(mobile="+7 (989) 111-12-33", home="+7 (465) 111-21-33"),
        "is_customer": True,
        "amount_paid": 450300.00,
        "customer_from": datetime(2021, 11, 1, 8, 0, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2026, 3, 10, 18, 45, tzinfo=timezone.utc),
    },
    {
        "id": 4,
        "first_name": "Елена",
        "last_name": "Сидорова",
        "email": "elena.sid@yandex.ru",
        "phone_numbers": Row(mobile="+7 (989) 110-12-33", home="+7 (465) 111-29-33"),
        "is_customer": True,
        "amount_paid": 1200.00,
        "customer_from": datetime(2025, 8, 24, 19, 12, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2025, 8, 24, 19, 30, tzinfo=timezone.utc),
    },
    {
        "id": 5,
        "first_name": "Алексей",
        "last_name": "Уваров",
        "email": "uvarov_alex@mail.ru",
        "phone_numbers": Row(mobile="+7 (989) 111-02-33", home="+7 (465) 511-21-33"),
        "is_customer": False,
        "amount_paid": 8900.75,
        "customer_from": datetime(2024, 2, 10, 11, 0, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2026, 4, 1, 12, 0, tzinfo=timezone.utc),
    },
]

In [3]:
df_users = spark.createDataFrame([Row(**user) for user in customers])

In [4]:
df_users.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_numbers: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- is_customer: boolean (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- customer_from: timestamp (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)



In [6]:
df_users.select("id", "phone_numbers.mobile").show()

+---+------------------+
| id|            mobile|
+---+------------------+
|  1|+7 (999) 111-22-33|
|  2|+7 (999) 444-55-66|
|  3| +1 (555) 019-2834|
|  4|              NULL|
|  5|+7 (903) 777-88-99|
+---+------------------+



In [12]:
from pyspark.sql.functions import col

df_users.select(
    "*",
    col("phone_numbers")["mobile"].alias("mobile"),
    col("phone_numbers")["home"].alias("home")
).drop("phone_numbers") \
 .show()

+---+----------+---------+--------------------+-----------+-----------+-------------------+-------------------+------------------+------------------+
| id|first_name|last_name|               email|is_customer|amount_paid|      customer_from|    last_updated_ts|            mobile|              home|
+---+----------+---------+--------------------+-----------+-----------+-------------------+-------------------+------------------+------------------+
|  1|      Иван|   Иванов|  ivanov@example.com|       true|    15450.5|2023-05-12 13:30:00|2026-01-15 17:22:00|+7 (999) 111-22-33|+7 (495) 111-22-33|
|  2|      Анна|  Петрова| petrova@example.com|      false|        0.0|               NULL|2026-02-20 12:15:00|+7 (999) 444-55-66|              NULL|
|  3|    Dmitry|    Smith|d.smith@global-te...|       true|   450300.0|2021-11-01 11:00:00|2026-03-10 21:45:00| +1 (555) 019-2834| +1 (555) 019-5555|
|  4|     Елена| Сидорова| elena.sid@yandex.ru|       true|     1200.0|2025-08-24 22:12:00|2025-08-2

In [13]:
spark.stop()